In [ ]:
from itables import init_notebook_mode, show
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

from aacbr import AACBR

init_notebook_mode(all_interactive=True)

## Data Set

In [ ]:
from ucimlrepo import fetch_ucirepo 
  
# fetch dataset 
connectionist_bench_sonar_mines_vs_rocks = fetch_ucirepo(id=151) 
  
# data (as pandas dataframes) 
X = connectionist_bench_sonar_mines_vs_rocks.data.features 
y = connectionist_bench_sonar_mines_vs_rocks.data.targets 

show(X)
print(y.nunique())



In [ ]:
encoder = LabelEncoder()
encoder.fit(y)
y = encoder.transform(y)


In [ ]:
print(encoder.classes_)
print(y)
print(type(y))

In [ ]:
X = X.values
print(X)

## Train Model

### Split into Training and Test Sets

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Test Size:  {len(X_test)}")
print(f"Total Train Size:  {len(X_train)}")
train_size = len(X_train)



In [ ]:
print(X_train)

### Build AF

In [ ]:
# TODO: Use more sophisticated orders

# Compare against the average for each column
means = X_train.mean(axis=0)
std = X_train.std(axis=0)

STD_PARAM = 2

def binarise_by_normal(case):
    return np.where(np.abs(case - means) <= STD_PARAM*std, 0, 1)


def binarise_by_normal_signed(case):
    return np.where(case - means <= -STD_PARAM*std, -1, np.where(case - means >= STD_PARAM*std, 1, 0))


def strictsuperset(a, b):

    if b.ndim == 1:
        b = np.expand_dims(b, axis = 0)

    return np.logical_and(np.all(a & b == b, axis = -1), np.logical_not(np.all(a & b == a, axis = -1)))


In [ ]:
COMPARISON_FUNC = strictsuperset
PREPROCESS_FUNC = binarise_by_normal 

In [ ]:
DEFAULT_OUTCOME = 0
DEFAULT_CASE = means.copy()
kf = KFold(n_splits=5, shuffle=True, random_state=42)

In [ ]:

def run_model(X_train, y_train, X_test, y_test, show_graph=False):
    X_train = PREPROCESS_FUNC(X_train)
    X_test = PREPROCESS_FUNC(X_test)
    default_case = PREPROCESS_FUNC(DEFAULT_CASE)
    
    model = AACBR(X_train, y_train, COMPARISON_FUNC, default_case, DEFAULT_OUTCOME)

    if show_graph:
        model.show_graph_with_labels()
    
    predicted = model(X_test)
    
    return([
        accuracy_score(y_test, predicted),
        precision_score(y_test, predicted),
        recall_score(y_test, predicted),
        f1_score(y_test, predicted)
    ])

### Cross-Validation

In [ ]:
metrics = []
for i, (train_index,  val_index) in enumerate(kf.split(X_train)):
    training_instances = X_train[train_index]
    training_labels = y_train[train_index]
    validation_instances = X_train[val_index]
    validation_labels = y_train[val_index]


    metrics.append(
        run_model(training_instances, training_labels, validation_instances, validation_labels)
    )

print("Accuracy, Precision, Recall, F1")
print(np.mean(metrics, axis=0))


### Test Set

In [ ]:
print("Accuracy, Precision, Recall, F1")
run_model(X_train, y_train, X_test, y_test, show_graph=False)